In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_SOURCE_CT = True
REUSE_FROZEN_LAD = True
REUSE_AORTA = True
REUSE_CALIBRATION = False
REUSE_TRACKING = False
REUSE_FIGURES = False
REUSE_REPORT = False


# OpenPlaque — Frozen LAD Proximal Reacquisition

Starts only from the independently frozen LAD proximal endpoint. No RCA, old trunk, or secondary centerline is loaded or used as a target. The aorta is only a stop/anatomical constraint.


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --depth 1 --branch lad-frozen-proximal-reacquisition-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy pandas matplotlib
import sys
sys.path.insert(0,'/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD


In [ ]:
from IPython.display import display, Image
from openplaque.lad_frozen_proximal_reacquisition import LADFrozenProximalReacquisitionWorkflow, synthetic_proximal_reacquisition_self_test
test=synthetic_proximal_reacquisition_self_test(); display(test); assert test['passed'], test
wf=LADFrozenProximalReacquisitionWorkflow(root='/content/drive/MyDrive/OpenPlaque', reuse={'source_ct':REUSE_SOURCE_CT,'frozen_lad':REUSE_FROZEN_LAD,'aorta':REUSE_AORTA,'calibration':REUSE_CALIBRATION,'tracking':REUSE_TRACKING,'figures':REUSE_FIGURES,'report':REUSE_REPORT})
display(wf.cache_status())


In [ ]:
wf.load_source_ct(); wf.load_frozen_lad(); wf.load_aorta(); cal=wf.calibrate_from_frozen_lad(); display(cal)
print('Frozen LAD points:', len(wf.frozen))


In [ ]:
summary=wf.run_tracking(); display(summary)
print('Accepted proximal track QC:'); display(wf.track_qc)
print('Top terminal rejection reasons:')
display(wf.terminal.reason.value_counts().rename_axis('reason').reset_index(name='count').head(20) if len(wf.terminal) else wf.terminal)


In [ ]:
names=wf.make_figures()
for name in names:
    print(name)
    display(Image(filename=str(wf.out/name)))


In [ ]:
report=wf.build_report(); zip_path=wf.package()
print('STATUS:', wf.summary['status'])
print('HTML:', report)
print('Final ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LAD_FROZEN_PROXIMAL_REACQUISITION_REPORT_BACK.zip')
